# **Revisor de Estatus de reportes por un elemento en comun**


*   El código toma hojas en especifico, en este caso serán: Fines, Propósitos y Componentes
*   Aísla los programas con 2 o más años sin revisión
*   Al igual que los programas sin ninguna revisión
*   Crea una hoja de resumen, una donde especifica los retrasados y otra donde incluye todos en el orden: Nula, vencida, vigente para cada dependencia
* Las hojas de excel se exportan en una carpeta





### Librerías

In [1]:
!pip install openpyxl
import sys # Manipulación de Timestamps
from datetime import datetime
import os # Crear la carpeta
import re # Limpieza de nombres
import openpyxl

### Configuración

In [7]:
# Archivos
archivo_entrada = "/content/ReporteMirComponente-20260804031314.xlsx"
carpeta_salida = "/content/reportes_finales"

# Generales
hojas = ["Fines", "Propositos", "Componentes"] # Hojas a revisar
tiempo = 2 # Años sin revisión MAC
datos = 5 # Los datos comienzan desde la fila 5

# Columnas que nos interesan
columnas = {
    6: "dependencia", 8: "nivel", 4: "programa", 5: "nombre_programa",
    9: "clave", 12: "clave_indicador", 21: "nombre_indicador", 18: "responsable",
    20: "correo", 19: "telefono", 104: "calificacion", 103: "capturista",
    102: "fecha",
}

# Columnas de salida
columnas_salida = ["nivel", "programa", "nombre_programa", "clave", "clave_indicador",
                   "nombre_indicador", "responsable", "correo", "telefono",
                   "calificacion", "capturista", "status", "comentario"]

# Encabezados
encabezados = ["Nivel", "Programa", "Nombre del programa", "Clave", "Clave Indicador",
               "Nombre del Indicador", "Responsable", "Correo", "Teléfono",
               "Ultima Calificación MAC", "Capturista MAC", "Status", "Comentario"]

# Convertir las fechas a string
def convertir_fecha(texto):
    for formato in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
        try:
            return datetime.strptime(str(texto).strip(), formato)
        except ValueError:
            pass
    return None


### Procesamiento de la base de datos y creación de carpeta de resultados

In [8]:
# Crea un archivo para cada dependencia limpio para evitar errores
def nombre_dep_archivo(texto):
  limpio = re.sub(r"[^A-Za-z0-9]+","_", str(texto)).strip("_")
  return limpio or "Sin_Dependencia"

libro = openpyxl.load_workbook(archivo_entrada, data_only=True)
hoy = datetime.now()
indicadores = []

for nombre_hoja in hojas:
    if nombre_hoja not in libro.sheetnames:
        continue
    hoja = libro[nombre_hoja]

    for fila in range(datos, hoja.max_row + 1):
        if hoja.cell(fila, 12).value is None:
            continue
        ind = {campo: hoja.cell(fila, col).value for col, campo in columnas.items()}
        ind["responsable"] = ind["responsable"] or "(sin nombre registrado)"
        ind["capturista"] = ind["capturista"] or "(sin capturista registrado)"

        fecha = convertir_fecha(ind["fecha"]) if ind["fecha"] else None
        if fecha is None:
            ind["anios"], ind["status"] = None, "Nula"
            ind["comentario"] = "Sin revisión registrada"
        else:
            anios = (hoy - fecha).days / 365.25
            ind["anios"] = round(anios, 1)
            ind["status"] = "Vencida" if anios >= tiempo else "Vigente"
            ind["comentario"] = f"Última revisión: {fecha:%d/%m/%Y} (hace {ind['anios']} años) — {ind['status']}."

        ind["retrasada"] = ind["status"] != "Vigente"
        indicadores.append(ind)

orden = {"Nula": 0, "Vencida": 1, "Vigente": 2}
indicadores.sort(key=lambda i: (orden[i["status"]], -(i["anios"] or 0)))

# Agrupar por dependencia
dependencias = {}
for i in indicadores:
    dep = dependencias.setdefault(nombre_dep_archivo(i["dependencia"]), [])
    dep.append(i)

# Un excel por cada dependencia
def escribir_hoja(libro, nombre, lista):
  hoja = libro.create_sheet(nombre)
  hoja.append(encabezados)
  for i in lista:
    hoja.append([i[campo] for campo in columnas_salida])

os.makedirs(carpeta_salida, exist_ok=True)

for dep, lista in dependencias.items():
  retrasados = [i for i in lista if i["retrasada"]]
  libro_salida = openpyxl.Workbook()
  libro_salida.remove(libro_salida.active)

  resumen = libro_salida.create_sheet("Resumen")
  resumen.append(["Análisis de retraso en revisión MAC"])
  resumen.append([])
  resumen.append(["Total de indicadores", len(lista)])
  resumen.append(["Retrasados (nula o vencida)", len(retrasados), f"{len(retrasados) / len(lista) * 100:.1f}%"])
  resumen.append([])
  resumen.append(["Nivel", "Total", "Retrasados"])

  conteo_por_nivel = {}
  for i in lista:
    c = conteo_por_nivel.setdefault(i["nivel"], {"total": 0, "retrasados": 0})
    c["total"] += 1
    c["retrasados"] += i["retrasada"]
  for nivel_i, c in conteo_por_nivel.items():
    resumen.append([nivel_i, c["total"], c["retrasados"]])

  escribir_hoja(libro_salida, "Retrasadas", retrasados)
  escribir_hoja(libro_salida, "Completas", lista)

  ruta = os.path.join(carpeta_salida, f"{nombre_dep_archivo(dep)}.xlsx")
  libro_salida.save(ruta)

print(f"Listo: se generaron {len(dependencias)} archivos en {carpeta_salida}")
print(f"Total de indicadores procesados: {len(indicadores)}")
print(f"Total retrasados: {sum(1 for i in indicadores if i['retrasada'])}")

Listo: se generaron 74 archivos en /content/reportes_finales
Total de indicadores procesados: 4319
Total retrasados: 3084


### Descargar la carpeta en zip

In [9]:
import shutil
from google.colab import files

# Comprime la carpeta completa en un archivo .zip
shutil.make_archive("reportes_por_dependencia", "zip", carpeta_salida)

# Descarga el .zip a la computadora
files.download("reportes_por_dependencia.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>